# 21 - Analyze two-model selective refinement (v2)

This notebook combines the existing uncertainty-only arms with the matched K=5 `(3,4)`
refine-last arms for both competent v2 models. The primary predeclared policy uses first-chunk
uncertainty: select the lower-U model, refine it for the episode only when U is at least `0.03`.

It can be run as a preview after shards 0-1: analysis is restricted to identities completed in
all four member/method arms. Set `REQUIRE_FULL_COHORT=True` only after shards 0-3 finish.

Prefix, individual-chunk, and full-episode summaries are also analyzed. Only the first-chunk rule
is directly deployable from these independently simulated trajectories. Later scores are useful
post-hoc evidence about predictiveness and headroom, but they are not presented as an online
policy. For `prefix_N`, trajectories with fewer than N chunks use their whole-episode uncertainty,
so every prefix sweep retains the entire matched cohort in its SR denominator. The best in-sample
window is exploratory and is evaluated on the same matched cohort used to choose it.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch and validate the exact four-arm cohort

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
from pnp.config import Method
from pnp.store import SupabaseStore
from pnp.diversity import (DIVERSITY_FIXED_REFINEMENT_THRESHOLD,
    DIVERSITY_PAIR_KEYS, DIVERSITY_V2_EXPERIMENT_PREFIX,
    analyze_diversity_selective_refinement,
    diversity_selective_refinement_figures, fetch_diversity_selective_refinement)

EXPERIMENT_PREFIX = DIVERSITY_V2_EXPERIMENT_PREFIX
FIXED_THRESHOLD = DIVERSITY_FIXED_REFINEMENT_THRESHOLD  # predeclared 0.03
REQUIRE_FULL_COHORT = False      # True only after shards 0,1,2,3 finish for both models
OUTPUT = Path("diversity_selective_refinement_v2_outputs")
OUTPUT.mkdir(exist_ok=True)

store = SupabaseStore()
all_rollouts, all_steps = fetch_diversity_selective_refinement(
    store, experiment_prefix=EXPERIMENT_PREFIX)
arm_counts = (all_rollouts.drop_duplicates(DIVERSITY_PAIR_KEYS + ["member_index", "method"])
              .groupby(DIVERSITY_PAIR_KEYS).size().rename("n_arms").reset_index())
complete_keys = arm_counts[arm_counts.n_arms == 4][DIVERSITY_PAIR_KEYS]
rollouts = all_rollouts.merge(complete_keys, on=DIVERSITY_PAIR_KEYS, validate="many_to_one")
steps = all_steps[all_steps.rollout_id.isin(rollouts.rollout_id)].copy()
counts = (rollouts.groupby(["member_index", "method"]).size()
          .rename("n").reset_index())
display(counts)
expected_methods = {Method.UNCERTAINTY, Method.REFINEMENT}
assert set(rollouts.method) == expected_methods
assert counts.n.nunique() == 1, "Four arms are not balanced after intersection"
N_PER_ARM = int(counts.n.iloc[0])
assert rollouts.suite.nunique() == 13
if REQUIRE_FULL_COHORT:
    assert N_PER_ARM == 1300, f"Full cohort requires 1,300/arm, found {N_PER_ARM}"
print({"analysis_mode": "FULL" if REQUIRE_FULL_COHORT else "PARTIAL PREVIEW",
       "identities_per_arm": N_PER_ARM,
       "ignored_incomplete_identity_rows": int((arm_counts.n_arms < 4).sum()),
       "rollouts": len(rollouts), "step_rows": len(steps),
       "experiments": sorted(rollouts.experiment.unique()),
       "fixed_threshold": FIXED_THRESHOLD})

## 3. Per-model baseline, refinement, AUC, and windows

In [ ]:
tables = analyze_diversity_selective_refinement(
    rollouts, steps, fixed_threshold=FIXED_THRESHOLD)
horizons = ["first_chunk", "prefix_2_chunks", "prefix_4_chunks",
            "prefix_8_chunks", "full_episode"]
member_overall = tables["member_refinement_overall"]
member_columns = ["member_index", "score_name", "n_pairs", "baseline_sr",
                  "refinement_sr", "delta_pp", "delta_ci_low_pp", "delta_ci_high_pp",
                  "F_to_S", "S_to_F", "failure_auc"]
for member_index in (0, 1):
    print(f"Model {member_index}: overall SR, matched refinement delta, and failure AUC")
    display(member_overall[(member_overall.member_index == member_index) &
                           member_overall.score_name.isin(horizons)][member_columns])
    print(f"Model {member_index}: top exploratory uncertainty windows")
    member_top = tables["member_refinement_top_windows"]
    display(member_top[(member_top.member_index == member_index) &
                       member_top.score_name.isin(horizons) & (member_top["rank"] <= 5)][[
        "score_name", "rank", "lower", "upper", "n_refined", "selective_sr",
        "delta_pp", "selected_F_to_S", "selected_S_to_F"]])

## 4. Match the shared source checkpoint on the same episodes

The v2 members started from `lerobot/pi05_libero_finetuned`. This loads that checkpoint's previous
K=5 `(3,4)` expanded-PRO rollouts and restricts them to the exact four-arm identity intersection.
Source baseline is the direct competence comparison; source refinement is included for context.

In [ ]:
from pnp.config import PI05_REPO_ID
from pnp.experiments import PRO_EXPANDED_EXPERIMENT
from pnp.diversity import analyze_checkpoint_refinement

source_runs = pd.DataFrame(store.fetch_all(
    "experiment_runs", "run_id,model_repo_id,model_revision",
    configure=lambda query: query.eq("experiment", PRO_EXPANDED_EXPERIMENT),
    order_by=("run_id",)))
display(source_runs[["model_repo_id", "model_revision"]].drop_duplicates())
assert set(source_runs.model_repo_id.dropna()) == {PI05_REPO_ID}

source_rows = pd.DataFrame(store.fetch_all(
    "rollouts", "rollout_id,suite,task_idx,episode_idx,init_state_hash,method,status,success,"
    "u_mean_episode,pnp_k,pnp_step_indices,refine_average",
    configure=lambda query: query.eq("experiment", PRO_EXPANDED_EXPERIMENT),
    order_by=("rollout_id",)))
source_rows = source_rows[
    source_rows.status.eq("completed") & source_rows.pnp_k.eq(5) &
    source_rows.pnp_step_indices.apply(lambda value: tuple(value or []) == (3, 4))]
source_observed_rows = source_rows[source_rows.method.eq(Method.UNCERTAINTY)].copy()
source_refined_rows = source_rows[
    source_rows.method.eq(Method.REFINEMENT) &
    ~source_rows.refine_average.fillna(False).astype(bool)].copy()
source_observed = source_observed_rows[
    DIVERSITY_PAIR_KEYS + ["success"]].rename(columns={"success": "source_baseline_success"})
source_refined = source_refined_rows[
    DIVERSITY_PAIR_KEYS + ["success"]].rename(columns={"success": "source_refinement_success"})
assert not source_observed.duplicated(DIVERSITY_PAIR_KEYS).any()
assert not source_refined.duplicated(DIVERSITY_PAIR_KEYS).any()

paired = tables["selective_refinement_paired_episodes"]
comparison = paired[DIVERSITY_PAIR_KEYS + ["success_observed_m0", "success_observed_m1"]].merge(
    source_observed, on=DIVERSITY_PAIR_KEYS, validate="one_to_one")
assert len(comparison), "No exact source-checkpoint episode matches were found"
comparison = comparison.merge(
    source_refined, on=DIVERSITY_PAIR_KEYS, how="left", validate="one_to_one")
print(f"Source checkpoint matched {len(comparison)}/{len(paired)} current identities exactly")
for column in ["success_observed_m0", "success_observed_m1", "source_baseline_success"]:
    comparison[column] = comparison[column].astype(bool)
comparison["source_refinement_success"] = comparison.source_refinement_success.astype("boolean")
source_overall = pd.DataFrame([{
    "n": len(comparison),
    "source_baseline_sr": comparison.source_baseline_success.mean(),
    "source_refinement_n": comparison.source_refinement_success.notna().sum(),
    "source_refinement_sr": comparison.source_refinement_success.mean(),
    "model0_baseline_sr": comparison.success_observed_m0.mean(),
    "model1_baseline_sr": comparison.success_observed_m1.mean(),
    "model0_delta_vs_source_pp": 100 * (
        comparison.success_observed_m0.mean() - comparison.source_baseline_success.mean()),
    "model1_delta_vs_source_pp": 100 * (
        comparison.success_observed_m1.mean() - comparison.source_baseline_success.mean()),
}])
source_by_suite = pd.DataFrame([{
    "suite": suite, "n": len(group),
    "source_baseline_sr": group.source_baseline_success.mean(),
    "source_refinement_n": group.source_refinement_success.notna().sum(),
    "source_refinement_sr": group.source_refinement_success.mean(),
    "model0_baseline_sr": group.success_observed_m0.mean(),
    "model1_baseline_sr": group.success_observed_m1.mean(),
} for suite, group in comparison.groupby("suite", sort=True)])
tables["source_checkpoint_matched_episodes"] = comparison
tables["source_checkpoint_overall"] = source_overall
tables["source_checkpoint_by_suite"] = source_by_suite
print("Matched shared-source checkpoint comparison")
display(source_overall)
display(source_by_suite)

# Run both window sweeps on the exact identities that have source baseline + refinement and all
# four current member/method arms. If any source refinement is missing, restrict the aggregation
# analysis too, so the source-versus-aggregation graph always uses identical episodes.
matched_keys = comparison[comparison.source_refinement_success.notna()][DIVERSITY_PAIR_KEYS]
assert len(matched_keys), "No exact episodes contain both source baseline and source refinement"
if len(matched_keys) != len(paired):
    matched_rollouts = rollouts.merge(matched_keys, on=DIVERSITY_PAIR_KEYS, validate="many_to_one")
    matched_steps = steps[steps.rollout_id.isin(matched_rollouts.rollout_id)].copy()
    tables = analyze_diversity_selective_refinement(
        matched_rollouts, matched_steps, fixed_threshold=FIXED_THRESHOLD)
    paired = tables["selective_refinement_paired_episodes"]
    print(f"Restricted aggregation window analysis to {len(paired)} exact source matches")
source_analysis_rollouts = pd.concat(
    [source_observed_rows, source_refined_rows], ignore_index=True).merge(
        matched_keys, on=DIVERSITY_PAIR_KEYS, validate="many_to_one")
source_observed_ids = source_analysis_rollouts[
    source_analysis_rollouts.method.eq(Method.UNCERTAINTY)].rollout_id.astype(str).tolist()
source_step_rows = []
for start in range(0, len(source_observed_ids), 100):
    batch = source_observed_ids[start:start + 100]
    source_step_rows.extend(store.fetch_all(
        "pnp_euler_steps", "rollout_id,chunk_idx,euler_step,u_mean",
        configure=lambda query, ids=batch: query.in_("rollout_id", ids),
        order_by=("rollout_id",)))
source_steps = pd.DataFrame(source_step_rows)
source_analysis = analyze_checkpoint_refinement(
    source_analysis_rollouts, source_steps, checkpoint_name="source_checkpoint")
for name, frame in source_analysis.items():
    tables[name.replace("member_refinement", "source_checkpoint_refinement")] = frame

source_horizons = source_analysis["member_refinement_overall"]
print("Shared source checkpoint: matched refinement delta and failure AUC")
display(source_horizons[source_horizons.score_name.isin(horizons)][[
    "score_name", "n_pairs", "baseline_sr", "refinement_sr", "delta_pp",
    "delta_ci_low_pp", "delta_ci_high_pp", "F_to_S", "S_to_F", "failure_auc"]])
print("Shared source checkpoint: top exploratory uncertainty windows")
source_top = source_analysis["member_refinement_top_windows"]
display(source_top[source_top.score_name.isin(horizons) & (source_top["rank"] <= 5)][[
    "score_name", "rank", "lower", "upper", "n_refined", "selective_sr",
    "delta_pp", "selected_F_to_S", "selected_S_to_F"]])

# Main comparison: independently optimize a window for the source checkpoint and for the
# two-model lower-U aggregation, then compare whole-cohort SR gains on the same identities.
aggregate_best = tables["selective_refinement_best_windows"]
aggregate_overall = tables["selective_refinement_overall"]
source_best = source_analysis["member_refinement_top_windows"]
source_overall_windows = source_analysis["member_refinement_overall"]
window_comparison = (
    source_best[(source_best["rank"] == 1) & source_best.score_name.isin(horizons)][[
        "score_name", "lower", "upper", "n_refined", "selective_sr", "delta_pp"]]
    .rename(columns={
        "lower": "source_lower", "upper": "source_upper",
        "n_refined": "source_n_refined", "selective_sr": "source_window_sr",
        "delta_pp": "source_window_delta_pp"})
    .merge(source_overall_windows[source_overall_windows.score_name.isin(horizons)][[
        "score_name", "baseline_sr"]].rename(columns={"baseline_sr": "source_baseline_sr"}),
        on="score_name", validate="one_to_one")
    .merge(aggregate_best[aggregate_best.score_name.isin(horizons)][[
        "score_name", "lower", "upper", "n_refined", "selective_sr", "delta_pp",
        "delta_vs_best_fixed_pp"]].rename(columns={
            "lower": "aggregate_lower", "upper": "aggregate_upper",
            "n_refined": "aggregate_n_refined", "selective_sr": "aggregate_window_sr",
            "delta_pp": "aggregate_window_delta_pp",
            "delta_vs_best_fixed_pp": "aggregate_window_vs_best_member_pp"}),
        on="score_name", validate="one_to_one")
    .merge(aggregate_overall[aggregate_overall.score_name.isin(horizons)][[
        "score_name", "lower_u_baseline_sr", "best_fixed_member_sr"]],
        on="score_name", validate="one_to_one"))
window_comparison["aggregate_delta_advantage_pp"] = (
    window_comparison.aggregate_window_delta_pp - window_comparison.source_window_delta_pp)
window_comparison["aggregate_window_vs_source_window_pp"] = 100 * (
    window_comparison.aggregate_window_sr - window_comparison.source_window_sr)
window_comparison["score_name"] = pd.Categorical(
    window_comparison.score_name, categories=horizons, ordered=True)
window_comparison = window_comparison.sort_values("score_name")
tables["source_vs_aggregate_best_windows"] = window_comparison

## 5. Main result: source model versus two-model aggregation

For each uncertainty horizon, both systems receive their own best lower/upper uncertainty window.
Every SR and delta uses the same matched episode cohort. `aggregate_delta_advantage_pp` is the key
column: positive means the two-model aggregation gains more SR from its window than the shared
source checkpoint gains from its own window.

In [ ]:
main_columns = [
    "score_name",
    "source_baseline_sr", "source_lower", "source_upper", "source_n_refined",
    "source_window_sr", "source_window_delta_pp",
    "lower_u_baseline_sr", "aggregate_lower", "aggregate_upper", "aggregate_n_refined",
    "aggregate_window_sr", "aggregate_window_delta_pp",
    "aggregate_delta_advantage_pp", "best_fixed_member_sr",
    "aggregate_window_vs_best_member_pp"]
display(window_comparison[main_columns].rename(columns={
    "score_name": "uncertainty_horizon",
    "lower_u_baseline_sr": "aggregate_no_refinement_sr"}))

## 6. Test alternative two-model uncertainty signals

The executed model is still whichever member has lower uncertainty. Only the score deciding
whether to refine changes: `minimum_u`, `(U0+U1)/2`, `maximum_u`, or `abs(U0-U1)`. Each signal gets
its own window sweep. Positive `delta_advantage_vs_source_pp` means that signal's best aggregation
window improves SR more than the source checkpoint's own best window.

In [ ]:
from pnp.diversity import aggregation_gate_signal_window_sweep

gate_sweep, gate_best = aggregation_gate_signal_window_sweep(
    tables["selective_refinement_policy_pairs"])
source_reference = window_comparison[[
    "score_name", "source_window_delta_pp", "source_window_sr"]]
gate_comparison = gate_best[gate_best.score_name.isin(horizons)][[
    "score_name", "gate_signal", "lower", "upper", "n_refined", "selective_sr",
    "delta_pp", "delta_vs_best_fixed_pp"]].merge(
        source_reference, on="score_name", validate="many_to_one")
gate_comparison["delta_advantage_vs_source_pp"] = (
    gate_comparison.delta_pp - gate_comparison.source_window_delta_pp)
gate_comparison["window_sr_vs_source_window_pp"] = 100 * (
    gate_comparison.selective_sr - gate_comparison.source_window_sr)
signal_order = ["minimum_u", "mean_u", "maximum_u", "absolute_u_gap"]
gate_comparison["score_name"] = pd.Categorical(
    gate_comparison.score_name, categories=horizons, ordered=True)
gate_comparison["gate_signal"] = pd.Categorical(
    gate_comparison.gate_signal, categories=signal_order, ordered=True)
gate_comparison = gate_comparison.sort_values(["score_name", "gate_signal"])
signal_winners = gate_comparison.sort_values(
    ["score_name", "delta_pp"], ascending=[True, False]).groupby(
        "score_name", observed=True).head(1)
tables["alternative_aggregate_gate_signal_sweep"] = gate_sweep
tables["alternative_aggregate_gate_signal_best_windows"] = gate_comparison
tables["alternative_aggregate_gate_signal_winners"] = signal_winners

display(gate_comparison.rename(columns={
    "score_name": "uncertainty_horizon",
    "selective_sr": "aggregate_window_sr",
    "delta_pp": "aggregate_window_delta_pp",
    "delta_vs_best_fixed_pp": "aggregate_window_vs_best_member_pp"}))
print("Best two-model gating signal at each horizon")
display(signal_winners[[
    "score_name", "gate_signal", "lower", "upper", "n_refined", "selective_sr",
    "delta_pp", "source_window_delta_pp", "delta_advantage_vs_source_pp"]])

## 7. Oracle ceilings and source-plus-member ensembles

The oracle rows are impossible selectors that know each rollout's outcome in advance. They answer
whether the recorded arms contain enough complementary successes for a real selector to exploit.
The four-arm member oracle succeeds if any of model 0/1 baseline/refinement succeeds; the six-arm
oracle also includes source baseline/refinement.

The source-plus-member experiments are implementable selection rules on the recorded trajectories:
choose the lower-uncertainty policy from `{source, model 0}` or `{source, model 1}`, then refine that
chosen policy only when its uncertainty falls inside the swept window. Positive
`delta_advantage_vs_source_pp` means the pair's window gain exceeds the source checkpoint's own
best window gain at that horizon.

In [ ]:
from pnp.diversity import analyze_source_member_ensembles

# Exact episode-level oracle ceilings. These are diagnostics, not deployable policies.
oracle_episodes = paired[DIVERSITY_PAIR_KEYS + [
    "success_observed_m0", "success_refined_m0",
    "success_observed_m1", "success_refined_m1"]].merge(
        comparison[DIVERSITY_PAIR_KEYS + [
            "source_baseline_success", "source_refinement_success"]],
        on=DIVERSITY_PAIR_KEYS, validate="one_to_one")
outcome_columns = [
    "source_baseline_success", "source_refinement_success",
    "success_observed_m0", "success_refined_m0",
    "success_observed_m1", "success_refined_m1"]
for column in outcome_columns:
    oracle_episodes[column] = oracle_episodes[column].astype(bool)

oracle_policies = {
    "source baseline": ["source_baseline_success"],
    "source refinement": ["source_refinement_success"],
    "model 0 baseline": ["success_observed_m0"],
    "model 1 baseline": ["success_observed_m1"],
    "2-member baseline oracle": ["success_observed_m0", "success_observed_m1"],
    "4-arm member oracle": ["success_observed_m0", "success_refined_m0",
                            "success_observed_m1", "success_refined_m1"],
    "source + members baseline oracle": ["source_baseline_success",
                                          "success_observed_m0", "success_observed_m1"],
    "all 6-arm oracle": outcome_columns,
}
source_baseline_sr = oracle_episodes.source_baseline_success.mean()
best_source_window_sr = window_comparison.source_window_sr.max()
oracle_summary = pd.DataFrame([{
    "policy": name, "n": len(oracle_episodes),
    "n_success": int(oracle_episodes[columns].any(axis=1).sum()),
    "success_rate": oracle_episodes[columns].any(axis=1).mean(),
    "delta_vs_source_baseline_pp": 100 * (
        oracle_episodes[columns].any(axis=1).mean() - source_baseline_sr),
    "delta_vs_best_source_window_pp": 100 * (
        oracle_episodes[columns].any(axis=1).mean() - best_source_window_sr),
} for name, columns in oracle_policies.items()])
tables["oracle_opportunity_episodes"] = oracle_episodes
tables["oracle_opportunity_summary"] = oracle_summary
print("Oracle opportunity (upper bounds; outcome knowledge is not deployable)")
display(oracle_summary)

# Source+m0 and source+m1: lower-U policy selection followed by a windowed refinement gate.
source_member = analyze_source_member_ensembles(
    tables["member_refinement_pairs"],
    source_analysis["member_refinement_pairs"],
    fixed_threshold=FIXED_THRESHOLD)
for name, frame in source_member.items():
    tables[name] = frame

source_member_best = source_member["source_member_best_windows"]
source_member_overall = source_member["source_member_overall"]
source_member_comparison = (
    source_member_best[source_member_best.score_name.isin(horizons)][[
        "ensemble", "member_index", "score_name", "lower", "upper", "n_refined",
        "selective_sr", "delta_pp", "delta_vs_best_fixed_pp"]]
    .merge(source_member_overall[source_member_overall.score_name.isin(horizons)][[
        "ensemble", "score_name", "lower_u_baseline_sr", "best_fixed_member_sr"]],
        on=["ensemble", "score_name"], validate="one_to_one")
    .merge(source_reference, on="score_name", validate="many_to_one"))
source_member_comparison["delta_advantage_vs_source_pp"] = (
    source_member_comparison.delta_pp - source_member_comparison.source_window_delta_pp)
source_member_comparison["window_sr_vs_source_window_pp"] = 100 * (
    source_member_comparison.selective_sr - source_member_comparison.source_window_sr)
source_member_comparison["score_name"] = pd.Categorical(
    source_member_comparison.score_name, categories=horizons, ordered=True)
source_member_comparison = source_member_comparison.sort_values(
    ["member_index", "score_name"])
tables["source_member_best_window_comparison"] = source_member_comparison
display(source_member_comparison[[
    "ensemble", "score_name", "lower_u_baseline_sr", "best_fixed_member_sr",
    "lower", "upper", "n_refined", "selective_sr", "delta_pp",
    "source_window_delta_pp", "delta_advantage_vs_source_pp",
    "source_window_sr", "window_sr_vs_source_window_pp"]].rename(columns={
        "score_name": "uncertainty_horizon",
        "best_fixed_member_sr": "best_fixed_pair_sr",
        "selective_sr": "source_member_window_sr",
        "delta_pp": "source_member_window_delta_pp"}))

## 8. Supporting aggregation details

The table below retains the five best windows per horizon so nearby alternatives can be inspected.
The fixed `0.03` gate is shown only as a secondary predeclared comparison; it is not the main
window-sweep result.

In [ ]:
top = tables["selective_refinement_top_windows"]
display(top[top.score_name.isin(horizons) & (top["rank"] <= 5)][[
    "score_name", "rank", "lower", "upper", "n_refined", "selective_sr",
    "delta_pp", "delta_vs_best_fixed_pp", "selected_F_to_S", "selected_S_to_F"]])

overall = tables["selective_refinement_overall"]
print("Secondary comparison: fixed U >= 0.03 gate")
display(overall[overall.score_name.isin(horizons)][[
    "score_name", "best_fixed_member_sr", "lower_u_baseline_sr",
    "fixed_threshold_sr", "fixed_delta_vs_lower_u_pp",
    "fixed_delta_vs_best_fixed_pp", "fixed_vs_best_ci_low_pp",
    "fixed_vs_best_ci_high_pp", "n_refined_fixed", "fixed_F_to_S", "fixed_S_to_F"]])

## 9. Save tables and figures

In [ ]:
for name, frame in tables.items():
    frame.to_csv(OUTPUT / f"{name}.csv", index=False)
paths = diversity_selective_refinement_figures(tables, OUTPUT / "figures")
primary_names = {"source_vs_aggregate_best_windows.png",
                 "alternative_aggregate_gate_signals.png",
                 "oracle_opportunity.png",
                 "source_member_best_windows.png"} | {
    f"source_vs_aggregate_window_sweep_{name}.png" for name in horizons}
print("Primary source-versus-aggregation figures")
for path in paths:
    if path.name in primary_names:
        print(path.name)
        display(Image(filename=str(path)))
print(f"Saved {len(paths) - len(primary_names)} additional diagnostic figures without displaying them.")

## 10. Concise readout

In [ ]:
for _, row in window_comparison.iterrows():
    print(str(row.score_name).replace("_", " "))
    print("  source window:    %5.1f%% SR (%+.2f pp over source baseline)" %
          (100 * row.source_window_sr, row.source_window_delta_pp))
    print("  aggregate window: %5.1f%% SR (%+.2f pp over lower-U baseline)" %
          (100 * row.aggregate_window_sr, row.aggregate_window_delta_pp))
    print("  aggregate delta advantage over source: %+.2f pp" %
          row.aggregate_delta_advantage_pp)
four_arm = oracle_summary[oracle_summary.policy.eq("4-arm member oracle")].iloc[0]
six_arm = oracle_summary[oracle_summary.policy.eq("all 6-arm oracle")].iloc[0]
print("\nOracle ceilings (not deployable)")
print("  four member arms: %.1f%% SR (%+.2f pp versus best source window)" %
      (100 * four_arm.success_rate, four_arm.delta_vs_best_source_window_pp))
print("  source + all member arms: %.1f%% SR (%+.2f pp versus best source window)" %
      (100 * six_arm.success_rate, six_arm.delta_vs_best_source_window_pp))
best_pair = source_member_comparison.sort_values(
    "window_sr_vs_source_window_pp", ascending=False).iloc[0]
print("Best source+member result")
print("  %s, %s: %.1f%% SR (%+.2f pp versus source's window at that horizon)" %
      (best_pair.ensemble, str(best_pair.score_name).replace("_", " "),
       100 * best_pair.selective_sr, best_pair.window_sr_vs_source_window_pp))
print("\nFirst chunk is deployable at the initial observation; later horizons are post-hoc here.")